# Domain of Validity (DoV)

Creates DoV models for the training set (to evaluate the test set) and for the full set to be used to evaluate DoV for **external** predictions

In [2]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path(PATH TO PBSG)))


import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [3]:
df = pd.read_csv('data_regressor.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [4]:
with open("random_stratified_splits_reg.json") as f:
    splits = json.load(f)

folds        = splits["folds"]
test_idx     = splits["test"]
all_idx      = np.arange(len(df))
trainval_idx = np.setdiff1d(all_idx, test_idx)
print(f"trainval={len(trainval_idx)}  test={len(test_idx)}  folds={len(folds)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

trainval=345  test=62  folds=5
Device: cpu


In [5]:
from dov import DomainOfValidity

In [6]:
# Fit DoV on trainval
dov = DomainOfValidity(k=5, threshold_percentile=5)
dov.fit(
    train_smiles = df.iloc[trainval_idx]["repeat_units"].tolist(),
)

# Score test set
result_df = dov.score_df(
    query_smiles = df.iloc[test_idx]["repeat_units"].tolist(),
)

print(result_df["in_domain_chem"].value_counts())
print(result_df["reliability"].value_counts())
print("\nOOD polymers:")
print(result_df[~result_df["in_domain_chem"]][["smiles","dov_score","reliability"]])

  DoV threshold (p5): 0.497
in_domain_chem
True     61
False     1
Name: count, dtype: int64
reliability
High        56
Moderate     5
Very low     1
Low          0
Name: count, dtype: int64

OOD polymers:
                                               smiles  dov_score reliability
47  [*]C1=CC=C(C(C2=CC=C(OC3=CC=C(O[*])C=C3)C=C2)=...      0.241    Very low


In [7]:
result_df.to_csv("regressor_test_dov_scores.csv", index=False)

In [8]:
import pickle

with open("dov_reg.pkl", "wb") as f:
    pickle.dump(dov, f)

In [9]:
# Fit DoV on full set
dov_full = DomainOfValidity(k=5, threshold_percentile=5)
dov_full.fit(
    train_smiles = df.iloc[all_idx]["repeat_units"].tolist(),
)


  DoV threshold (p5): 0.558


In [10]:
import pickle

with open("dov_reg_full.pkl", "wb") as f:
    pickle.dump(dov_full, f)